In [8]:

#task 1
import random


def evaluateExpression(x, y, z):
    return 6 * x ** 3 + 9 * y ** 2 + 90 * z - 25

def fitness(x, y, z):
    ans = evaluateExpression(x, y, z)
    if ans == 0:
        return 99999
    else:
        return abs(1 / ans)


solutions = []
for _ in range(1000):
    solutions.append((
        random.uniform(0, 10),
        random.uniform(0, 10),
        random.uniform(0, 10)
    ))

for generation_count in range(10000):
    rankedSolutions = []
    for solution in solutions:
        rankedSolutions.append((fitness(solution[0], solution[1], solution[2]), solution))
    rankedSolutions.sort()
    rankedSolutions.reverse()

    if rankedSolutions[0][0] > 999:
        best = rankedSolutions[0][1]
        print(f"Solution found at generation {generation_count}")
        print(f"x = {best[0]:.4f}")
        print(f"y = {best[1]:.4f}")
        print(f"z = {best[2]:.4f}")
        print(f"Expression value = {evaluateExpression(*best):.6f}")
        break

    bestSolution = rankedSolutions[:100]
    variables = []
    for s in bestSolution:
        variables.append(s[1][0])
        variables.append(s[1][1])
        variables.append(s[1][2])

    newGeneration = []
    for _ in range(1000):
        x = random.choice(variables) * random.uniform(0.99, 1.01)
        y = random.choice(variables) * random.uniform(0.99, 1.01)
        z = random.choice(variables) * random.uniform(0.99, 1.01)
        newGeneration.append((x, y, z))
    solutions = newGeneration


Solution found at generation 5
x = 0.1606
y = 0.2689
z = 0.2703
Expression value = -0.000442


In [5]:
#task 2

import random

N = 5
POPULATION_SIZE = 100
MAX_GENERATIONS = 10000
MUTATION_RATE = 0.1
MAX_FITNESS = N * (N - 1) // 2


def generate_chromosome():
    return [random.randint(1, N) for _ in range(N)]

def generate_population(size):
    return [generate_chromosome() for _ in range(size)]

def fitness_queens(chromosome):
    non_attacking = 0
    for i in range(N):
        for j in range(i + 1, N):
            if chromosome[i] != chromosome[j]:
                if abs(chromosome[i] - chromosome[j]) != abs(i - j):
                    non_attacking += 1
    return non_attacking

def selection(population):
    scored = [(fitness_queens(c), c) for c in population]
    scored.sort(reverse=True)
    total = sum(s for s, _ in scored)
    if total == 0:
        return random.choice(population)
    pick = random.uniform(0, total)
    current = 0
    for score, chrom in scored:
        current += score
        if current >= pick:
            return chrom
    return scored[0][1]

def crossover(parent1, parent2):
    point = random.randint(1, N - 1)
    child1 = parent1[:point] + parent2[point:]
    child2 = parent2[:point] + parent1[point:]
    return child1, child2


def mutate(chromosome):
    if random.random() < MUTATION_RATE:
        idx = random.randint(0, N - 1)
        chromosome[idx] = random.randint(1, N)
    return chromosome

def print_board(chromosome):
    print("\nBoard Configuration:")
    for row in range(N, 0, -1):
        line = ""
        for col in range(N):
            if chromosome[col] == row:
                line += " Q "
            else:
                line += " . "
        print(f"{row} |{line}|")
    print("    " + "  ".join(str(i+1) for i in range(N)))
    print(f"\nChromosome: {chromosome}")
    print(f"Fitness: {fitness_queens(chromosome)} / {MAX_FITNESS}")


population = generate_population(POPULATION_SIZE)
solution_found = False

for gen in range(MAX_GENERATIONS):
    population.sort(key=fitness_queens, reverse=True)
    best = population[0]
    best_fitness = fitness_queens(best)

    if best_fitness == MAX_FITNESS:
        print(f"Solution found at generation {gen}!")
        print_board(best)
        solution_found = True
        break

    new_population = population[:10]
    while len(new_population) < POPULATION_SIZE:
        p1 = selection(population)
        p2 = selection(population)
        c1, c2 = crossover(p1[:], p2[:])
        new_population.append(mutate(c1))
        if len(new_population) < POPULATION_SIZE:
            new_population.append(mutate(c2))
    population = new_population

if not solution_found:
    population.sort(key=fitness_queens, reverse=True)
    print("Best solution found:")
    print_board(population[0])


Solution found at generation 4!

Board Configuration:
5 | Q  .  .  .  . |
4 | .  .  Q  .  . |
3 | .  .  .  .  Q |
2 | .  Q  .  .  . |
1 | .  .  .  Q  . |
    1  2  3  4  5

Chromosome: [5, 2, 4, 1, 3]
Fitness: 10 / 10


In [ ]:
#task 3
import random

items = [
    {'name': 'A', 'weight': 2, 'value': 6},
    {'name': 'B', 'weight': 3, 'value': 10},
    {'name': 'C', 'weight': 1, 'value': 3},
    {'name': 'D', 'weight': 4, 'value': 12},
    {'name': 'E', 'weight': 2, 'value': 7},
]
CAPACITY = 7
NUM_ITEMS = len(items)
POPULATION_SIZE = 100
MAX_GENERATIONS = 500
MUTATION_RATE = 0.05


def generate_chromosome_ks():
    return [random.randint(0, 1) for _ in range(NUM_ITEMS)]

def generate_population_ks(size):
    return [generate_chromosome_ks() for _ in range(size)]


def fitness_ks(chromosome):
    total_weight = sum(chromosome[i] * items[i]['weight'] for i in range(NUM_ITEMS))
    total_value  = sum(chromosome[i] * items[i]['value']  for i in range(NUM_ITEMS))
    if total_weight > CAPACITY:
        return 0
    return total_value


def selection_ks(population):
    scored = [(fitness_ks(c), c) for c in population]
    total = sum(s for s, _ in scored)
    if total == 0:
        return random.choice(population)
    pick = random.uniform(0, total)
    current = 0
    for score, chrom in scored:
        current += score
        if current >= pick:
            return chrom
    return scored[-1][1]


def crossover_ks(p1, p2):
    point = random.randint(1, NUM_ITEMS - 1)
    return p1[:point] + p2[point:], p2[:point] + p1[point:]


def mutate_ks(chromosome):
    for i in range(NUM_ITEMS):
        if random.random() < MUTATION_RATE:
            chromosome[i] = 1 - chromosome[i]
    return chromosome

population = generate_population_ks(POPULATION_SIZE)

for gen in range(MAX_GENERATIONS):
    population.sort(key=fitness_ks, reverse=True)
    new_population = population[:10]
    while len(new_population) < POPULATION_SIZE:
        p1 = selection_ks(population)
        p2 = selection_ks(population)
        c1, c2 = crossover_ks(p1[:], p2[:])
        new_population.append(mutate_ks(c1))
        if len(new_population) < POPULATION_SIZE:
            new_population.append(mutate_ks(c2))
    population = new_population

population.sort(key=fitness_ks, reverse=True)
best = population[0]
total_weight = sum(best[i] * items[i]['weight'] for i in range(NUM_ITEMS))
total_value  = sum(best[i] * items[i]['value']  for i in range(NUM_ITEMS))

print(f"Best chromosome: {best}")
print(f"Selected items: {[items[i]['name'] for i in range(NUM_ITEMS) if best[i] == 1]}")
print(f"Total value  = {total_value}")
print(f"Total weight = {total_weight}")


Best chromosome: [1, 1, 0, 0, 1]
Selected items: ['A', 'B', 'E']
Total value  = 23
Total weight = 7
